# 🧠 AIOS LoRA Fine-Tuning (Unsloth / PEFT)

Обучение собственного LoRA-адаптера на **Qwen2.5-7B-Instruct** (или Llama-3.1-8B) на базе датасета AIOS.

**Среда выполнения → T4 GPU** (Unsloth оптимизирован для Colab).

Датасет: загрузите `aios_coder_hf.jsonl` (с VPS, папка `data/finetune/`) в сессию Colab.

In [ ]:
!pip install -q unsloth
import torch
from unsloth import FastLanguageModel
print('✅ Unsloth установлен, CUDA:', torch.cuda.is_available())

In [ ]:
# === ЯЧЕЙКА 2: Загрузка датасета ===
import json, os
with open('aios_coder_hf.jsonl') as f:
    dataset = [json.loads(l) for l in f if l.strip()]
print('✅ Датасет:', len(dataset), 'примеров')
print(dataset[0]['messages'][0]['content'][:80])

In [ ]:
# === ЯЧЕЙКА 3: Загрузка базовой модели (4bit LoRA) ===
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing=True,
)
print('✅ Модель + LoRA готовы')

In [ ]:
# === ЯЧЕЙКА 4: Форматирование + обучение ===
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

def fmt(ex):
    m = ex['messages']
    return {'text': tokenizer.apply_chat_template(m, tokenize=False)}
ds = Dataset.from_list(dataset).map(fmt)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    dataset_text_field='text', max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1, output_dir='outputs',
    ),
)
trainer.train()
print('✅ LoRA обучена')

In [ ]:
# === ЯЧЕЙКА 5: Сохранение LoRA + слияние ===
FastLanguageModel.for_inference(model)
model.save_pretrained('lora_model')
tokenizer.save_pretrained('lora_model')
# Слияние LoRA с базовой моделью для инференса
model = model.merge_and_unload()
model.save_pretrained('merged_model')
tokenizer.save_pretrained('merged_model')
print('✅ Сохранено: lora_model/ и merged_model/')
print('   Загрузите папку на VPS (или HF Hub) для инференса.')

In [ ]:
# === ЯЧЕЙКА 6: Проверка генерации ===
FastLanguageModel.for_inference(model)
msgs = [{'role':'user','content':'Fix a security issue: hard-coded API key in aios_core. Give Python code.'}]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=120)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))